# Suite PYPRT — Printing Python

The versions print any Python syntax node: a module as a file, anything else as the text it stands for. Trees built by hand
print as source text without help: parentheses are added where Python's grammar needs them, which is also where an
assignment expression, a `yield` or a lambda must be parenthesized. Printing validates the tree and checks it against
the version first.

In [ ]:
from mbse.Programs.Framework.Errors import PrintError
from mbse.Programs.Python import Python312, Python314, Syntax as S
from mbse.Programs.Python._Standard import PythonStandard
from support import raises

PY315 = PythonStandard(3, 15)
P = PY315.print


def n(spelling):
    return S.Identifier(spelling=spelling)


def v(spelling):
    return S.Name(id=n(spelling))


def c(spelling):
    return S.Constant(spelling=spelling)


def b(left, op, right):
    return S.BinOp(left=left, op=op, right=right)


def u(op, operand):
    return S.UnaryOp(op=op, operand=operand)


def pas():
    return [S.Pass()]


def expr(value):
    return S.Expr(value=value)


def compare():
    return S.Compare(left=v("a"), comparisons=[S.Comparison(op="<", comparator=v("b"))])


def both():
    return S.BoolOp(op="and", values=[v("a"), v("b")])


def lam():
    return S.Lambda(args=S.Arguments(), body=v("a"))


def ifexp():
    return S.IfExp(body=v("a"), test=v("b"), orelse=v("c"))


def walrus():
    return S.NamedExpr(target=v("x"), value=v("a"))


def yielded():
    return S.Yield(value=v("a"))


def pair():
    return S.Tuple(elts=[v("a"), v("b")])


def function():
    return S.FunctionDef(name=n("f"), args=S.Arguments(), body=pas())


def alternatives():
    return S.MatchOr(patterns=[S.MatchValue(value=c("1")), S.MatchValue(value=c("2"))])


def captured():
    return S.MatchAs(pattern=S.MatchAs(name=n("a")), name=n("b"))


def open_sequence():
    return S.MatchSequence(patterns=[S.MatchAs(name=n("a"))])

## PYPRT-01 · Precedence: operands are parenthesized where they bind less than their place needs

In [ ]:
assert P(b(b(v("a"), "+", v("b")), "*", v("c"))) == "(a + b) * c"
assert P(b(v("a"), "*", b(v("b"), "+", v("c")))) == "a * (b + c)"
assert P(b(v("a"), "-", b(v("b"), "-", v("c")))) == "a - (b - c)" and P(b(b(v("a"), "-", v("b")), "-", v("c"))) == "a - b - c"
assert P(b(v("a"), "**", b(v("b"), "**", v("c")))) == "a ** b ** c"
assert P(b(b(v("a"), "**", v("b")), "**", v("c"))) == "(a ** b) ** c"
assert P(b(u("-", v("a")), "**", v("b"))) == "(-a) ** b" and P(u("-", b(v("a"), "**", v("b")))) == "-a ** b"
assert P(b(v("a"), "**", u("-", v("b")))) == "a ** -b"
assert P(b(S.Await(value=v("a")), "**", v("b"))) == "await a ** b"
assert P(S.Await(value=b(v("a"), "**", v("b")))) == "await (a ** b)" and P(S.Await(value=S.Await(value=v("a")))) == "await (await a)"
assert P(b(b(v("a"), "^", v("b")), "&", v("c"))) == "(a ^ b) & c" and P(b(v("a"), "|", b(v("b"), "^", v("c")))) == "a | b ^ c"
assert P(u("not", u("not", v("a")))) == "not not a" and P(u("-", u("not", v("a")))) == "-(not a)"
assert P(S.Compare(left=compare(), comparisons=[S.Comparison(op="is not", comparator=compare())])) == "(a < b) is not (a < b)"
assert P(u("not", compare())) == "not a < b"
assert P(S.BoolOp(op="and", values=[both(), v("c")])) == "(a and b) and c"
assert P(S.BoolOp(op="or", values=[both(), u("not", v("c"))])) == "a and b or not c"
assert P(S.BoolOp(op="and", values=[S.BoolOp(op="or", values=[v("a"), v("b")]), v("c")])) == "(a or b) and c"
assert P(S.IfExp(body=ifexp(), test=ifexp(), orelse=ifexp())) == "(a if b else c) if (a if b else c) else a if b else c"
assert P(S.IfExp(body=lam(), test=v("b"), orelse=lam())) == "(lambda: a) if b else lambda: a"
assert P(S.Lambda(args=S.Arguments(), body=ifexp())) == "lambda: a if b else c"
assert P(expr(walrus())) == "(x := a)" and P(S.Assign(targets=[v("y")], value=walrus())) == "y = (x := a)"
assert P(S.If(test=walrus(), body=pas())) == "if x := a:\n    pass"
assert P(S.Call(func=v("f"), args=[walrus()], keywords=[S.Keyword(arg=n("k"), value=walrus())])) == "f(x := a, k=(x := a))"
assert P(S.NamedExpr(target=v("x"), value=S.NamedExpr(target=v("y"), value=v("a")))) == "x := (y := a)"
assert P(S.List(elts=[walrus()])) == "[x := a]" and P(S.Tuple(elts=[walrus(), v("b")])) == "(x := a), b"
assert P(S.Parenthesized(value=S.Tuple(elts=[walrus(), v("b")]))) == "(x := a, b)"
assert P(expr(yielded())) == "yield a" and P(S.Assign(targets=[v("x")], value=yielded())) == "x = yield a"
assert P(S.Return(value=yielded())) == "return (yield a)" and P(S.Call(func=v("f"), args=[yielded()])) == "f((yield a))"
assert P(S.Parenthesized(value=yielded())) == "(yield a)" and P(S.Return(value=S.Tuple(elts=[v("a"), v("b")]))) == "return a, b"
assert P(S.Call(func=v("f"), args=[pair()])) == "f((a, b))" and P(S.Tuple(elts=[pair()])) == "(a, b),"
assert P(S.Starred(value=b(v("a"), "+", v("b")))) == "*a + b" and P(S.Starred(value=both())) == "*(a and b)"
assert P(S.Attribute(value=b(v("a"), "+", v("b")), attr=n("c"))) == "(a + b).c"
assert P(S.Attribute(value=c("1"), attr=n("real"))) == "(1).real" and P(S.Attribute(value=c("1.0"), attr=n("real"))) == "1.0.real"
assert P(S.Subscript(value=S.Call(func=v("f")), slice=pair())) == "f()[a, b]"
assert P(S.Subscript(value=v("a"), slice=S.Tuple(elts=[v("i")]))) == "a[i,]"
assert P(S.Subscript(value=v("a"), slice=S.Tuple())) == "a[()]"
print("ok")

## PYPRT-02 · Every kind prints on its own, as the text it stands for

In [ ]:
gen = S.Comprehension(target=v("x"), iter=v("y"), ifs=[v("x"), u("not", v("x"))])
args = S.Arguments(posonlyargs=[S.Arg(arg=n("a"))], args=[S.Arg(arg=n("b"), annotation=v("int"), default_value=c("1"))],
                   vararg=S.Arg(arg=n("c")), kwonlyargs=[S.Arg(arg=n("d"), default_value=c("2"))],
                   kwarg=S.Arg(arg=n("e")))
texts = {
    n("x"): "x", S.DottedName(names=[n("a"), n("b")]): "a.b", c("0x_FF"): "0x_FF",
    S.ConcatenatedString(values=[c("'a'"), c('"b"')]): "'a' \"b\"",
    S.StringText(spelling="{{a}}"): "{{a}}", S.FormatSpec(values=[S.StringText(spelling=">10")]): ":>10",
    S.FormattedValue(value=v("x")): "{x}", S.Interpolation(value=v("x"), conversion="r"): "{x!r}",
    S.JoinedStr(prefix="f", quote="'", values=[S.StringText(spelling="a")]): "f'a'",
    S.TemplateStr(prefix="t", quote='"""', values=[]): 't""""""',
    S.Parenthesized(value=v("a")): "(a)", S.Tuple(): "()", S.Tuple(elts=[v("a")]): "a,", S.List(): "[]",
    S.Set(elts=[v("a")]): "{a}", S.Dict(): "{}", S.DictItem(key=v("k"), value=v("v")): "k: v",
    S.DictItem(value=v("m")): "**m", gen: "for x in y if x if not x",
    S.Comprehension(is_async=True, target=pair(), iter=pair()): "async for a, b in (a, b)",
    S.ListComp(elt=v("x"), generators=[S.Comprehension(target=v("x"), iter=v("y"))]): "[x for x in y]",
    S.SetComp(elt=v("x"), generators=[S.Comprehension(target=v("x"), iter=v("y"))]): "{x for x in y}",
    S.DictComp(key=v("k"), value=v("v"), generators=[S.Comprehension(target=v("k"), iter=v("y"))]): "{k: v for k in y}",
    S.DictComp(key=v("d"), generators=[S.Comprehension(target=v("d"), iter=v("y"))]): "{**d for d in y}",
    S.GeneratorExp(elt=v("x"), generators=[S.Comprehension(target=v("x"), iter=ifexp())]): "(x for x in (a if b else c))",
    S.Yield(): "yield", S.YieldFrom(value=v("a")): "yield from a",
    S.Slice(): ":", S.Slice(lower=v("a"), step=v("c")): "a::c", S.Slice(upper=v("b")): ":b",
    S.Keyword(arg=n("k"), value=v("v")): "k=v", S.Keyword(value=v("m")): "**m",
    S.Call(func=v("f"), args=[S.GeneratorExp(elt=v("x"), generators=[S.Comprehension(target=v("x"), iter=v("y"))])]):
        "f(x for x in y)",
    S.Call(func=v("f"), args=[S.GeneratorExp(elt=v("x"), generators=[S.Comprehension(target=v("x"), iter=v("y"))])],
           keywords=[S.Keyword(arg=n("k"), value=v("v"))]): "f((x for x in y), k=v)",
    S.Comparison(op="not in", comparator=v("a")): "not in a",
    args: "a, /, b: int = 1, *c, d=2, **e",
    S.Lambda(args=S.Arguments(args=[S.Arg(arg=n("a"), default_value=c("1"))], vararg=S.Arg(arg=n("b"))), body=v("a")):
        "lambda a=1, *b: a",
    S.Arguments(kwonlyargs=[S.Arg(arg=n("d"))]): "*, d", S.Arguments(): "",
    S.Arg(arg=n("a"), annotation=v("int")): "a: int", S.Alias(): "*",
    S.Alias(name=S.DottedName(names=[n("a"), n("b")]), asname=n("c")): "a.b as c",
    S.WithItem(context_expr=v("a"), optional_vars=pair()): "a as (a, b)",
    S.TypeVar(name=n("T"), bound=v("int"), default_value=v("str")): "T: int = str",
    S.ParamSpec(name=n("P"), default_value=v("a")): "**P = a", S.TypeVarTuple(name=n("Ts")): "*Ts",
    S.MatchValue(value=c("1")): "1", S.MatchSingleton(value="None"): "None", S.MatchStar(): "*_",
    S.MatchAs(): "_", S.MatchAs(name=n("a")): "a", S.MatchSequence(): "()",
    S.MatchMapping(keys=[c("1")], patterns=[S.MatchAs()], rest=n("r")): "{1: _, **r}",
    S.MatchClass(cls=v("P"), patterns=[S.MatchAs()], kwd_attrs=[n("x")], kwd_patterns=[S.MatchStar(name=n("s"))]):
        "P(_, x=*s)",
    S.Comment(text=" a"): "# a", S.Pass(): "pass", S.Break(): "break", S.Continue(): "continue",
    S.Global(names=[n("a"), n("b")]): "global a, b", S.Nonlocal(names=[n("a")]): "nonlocal a",
    S.Import(is_lazy=True, names=[S.Alias(name=S.DottedName(names=[n("a")]))]): "lazy import a",
    S.ImportFrom(is_lazy=True, level=2, names=[S.Alias()]): "lazy from .. import *",
    S.Raise(): "raise", S.Raise(exc=v("E"), cause=v("c")): "raise E from c",
    S.Delete(targets=[v("a"), S.Subscript(value=v("b"), slice=c("0"))]): "del a, b[0]",
    S.AnnAssign(target=v("a"), annotation=lam()): "a: lambda: a",
    S.AugAssign(target=v("a"), op="<<", value=yielded()): "a <<= yield a",
    S.TypeAlias(name=v("A"), value=v("int")): "type A = int",
    S.ExceptHandler(body=pas()): "except:\n    pass",
    S.MatchCase(pattern=S.MatchAs(), guard=walrus(), body=pas()): "case _ if x := a:\n    pass",
    S.Module(): "",
}
for node, text in texts.items():
    assert P(node) == text, (node, P(node))
print(len(texts), "texts")

## PYPRT-03 · Statements: one per line, blocks indented, `elif`, blank lines around definitions, and comments

In [ ]:
klass = S.ClassDef(decorator_list=[v("d")], name=n("C"), body=[expr(c("'doc'")), S.FunctionDef(
    name=n("m"), args=S.Arguments(args=[S.Arg(arg=n("self"))]), body=pas()), S.Pass()])
module = S.Module(body=[S.Import(names=[S.Alias(name=S.DottedName(names=[n("os")]))]), S.Comment(text=" lead"),
                        function(), klass, S.Comment(text=" trailing", trailing=True), S.Assign(targets=[v("x")],
                                                                                                value=c("1"))])
assert P(module) == ("import os\n\n\n# lead\ndef f():\n    pass\n\n\n@d\nclass C:\n    'doc'\n\n"
                     "    def m(self):\n        pass\n\n    pass  # trailing\n\n\nx = 1\n")
chain = S.If(test=v("a"), body=pas(), orelse=[S.If(test=v("b"), body=[S.Comment(text=" c", trailing=True), S.Pass()],
                                                   orelse=[S.If(test=v("c"), body=pas())])])
assert P(chain) == "if a:\n    pass\nelif b:  # c\n    pass\nelif c:\n    pass"
nested = S.If(test=v("a"), body=pas(), orelse=[S.Comment(text=" keeps else"), S.If(test=v("b"), body=pas())])
assert P(nested) == "if a:\n    pass\nelse:\n    # keeps else\n    if b:\n        pass"
loops = [S.While(test=v("a"), body=pas(), orelse=pas()), S.For(target=v("x"), iter=pair(), body=pas()),
         S.AsyncFor(target=v("x"), iter=v("y"), body=pas(), orelse=pas()), S.While(test=v("a"), body=pas())]
assert [P(l) for l in loops] == ["while a:\n    pass\nelse:\n    pass", "for x in a, b:\n    pass",
                                 "async for x in y:\n    pass\nelse:\n    pass", "while a:\n    pass"]
tried = S.TryStar(body=pas(), handlers=[S.ExceptHandler(type=pair(), body=pas()), S.ExceptHandler(
    type=pair(), name=n("e"), body=pas())], orelse=pas(), finalbody=pas())
assert P(tried) == ("try:\n    pass\nexcept* a, b:\n    pass\nexcept* (a, b) as e:\n    pass\nelse:\n    pass\n"
                    "finally:\n    pass")
assert P(S.With(items=[S.WithItem(context_expr=v("a"))], body=pas())) == "with a:\n    pass"
assert P(S.AsyncWith(items=[S.WithItem(context_expr=lam(), optional_vars=v("b"))], body=pas())) == (
    "async with lambda: a as b:\n    pass")
generic = S.AsyncFunctionDef(name=n("f"), type_params=[S.TypeVar(name=n("T"))], args=S.Arguments(), returns=v("T"),
                             body=pas())
assert P(generic) == "async def f[T]() -> T:\n    pass"
assert P(S.ClassDef(name=n("C"), bases=[v("B")], keywords=[S.Keyword(arg=n("m"), value=v("M"))], body=pas())) == (
    "class C(B, m=M):\n    pass")
assert P(S.Match(subject=pair(), cases=[S.MatchCase(pattern=S.MatchSequence(patterns=[S.MatchAs(), S.MatchAs()]),
                                                   body=pas())])) == "match a, b:\n    case _, _:\n        pass"
assert P(S.Module(body=[S.Comment(text=" only", trailing=True)])) == "# only\n"
assert P(S.Module(body=[S.Comment(text=" first"), function()])) == "# first\ndef f():\n    pass\n"
assert P(S.Module(body=[S.Assign(targets=[v("x")], value=c("1")), S.Comment(text=" t", trailing=True),
                        S.Comment(text=" lead"), function()])) == "x = 1  # t\n\n\n# lead\ndef f():\n    pass\n"
print("ok")

## PYPRT-04 · f-strings: a replacement field prints its text while that spells its value, and its value otherwise

In [ ]:
def joined(*values):
    return S.JoinedStr(prefix="f", quote="'", values=list(values))


field = S.FormattedValue(value=b(v("a"), "+", v("b")), text="a+b")
assert P(joined(field)) == "f'{a+b}'"
field.value = v("c")  # the text no longer spells it
assert P(joined(field)) == "f'{c}'"
debug = S.FormattedValue(value=v("x"), text=" x = ", debug=True, conversion="r",
                         format_spec=S.FormatSpec(values=[S.StringText(spelling=">"), S.FormattedValue(value=v("w"))]))
assert P(joined(debug)) == "f'{ x = !r:>{w}}'"
debug.text = "x = # a comment\n"
assert P(joined(debug)) == "f'{x = # a comment\n!r:>{w}}'"
debug.text, debug.value = " y = ", v("x")
assert P(joined(debug)) == "f'{x=!r:>{w}}'"
assert P(joined(S.FormattedValue(value=lam()))) == "f'{(lambda: a)}'" and P(joined(S.FormattedValue(value=walrus()))) == "f'{(x := a)}'"
assert P(joined(S.FormattedValue(value=S.Set(elts=[v("a")])))) == "f'{ {a}}'"
assert P(joined(S.FormattedValue(value=pair()))) == "f'{a, b}'" and P(joined(S.FormattedValue(value=yielded()))) == "f'{yield a}'"
assert P(joined(S.FormattedValue(value=v("x"), text="(x)"))) == "f'{x}'"
commented = S.FormattedValue(value=c('"""a # b"""'), text=' # a comment\n  """a # b"""=', debug=True)
assert P(S.JoinedStr(prefix="f", quote='"', values=[commented])) == 'f"{ # a comment\n  \"\"\"a # b\"\"\"=}"'
spaced = S.FormattedValue(value=c("'a b'"), text="'ab'")  # whitespace in a string is code
assert P(S.JoinedStr(prefix="f", quote='"', values=[spaced])) == "f\"{'a b'}\""
escaped = S.FormattedValue(value=c("'a\\'b'"), text=" 'a\\'b' ")
assert P(S.JoinedStr(prefix="f", quote='"', values=[escaped])) == "f\"{ 'a\\'b' }\""
print("ok")

## PYPRT-05 · Patterns: parentheses where a pattern needs them, and open sequences only in a case

In [ ]:
def case(pattern):
    return P(S.Match(subject=v("x"), cases=[S.MatchCase(pattern=pattern, body=pas())])).splitlines()[1][9:-1]


assert case(S.MatchAs(pattern=alternatives(), name=n("s"))) == "1 | 2 as s"
assert case(S.MatchAs(pattern=captured(), name=n("c"))) == "(a as b) as c"
assert case(S.MatchAs(pattern=alternatives())) == "1 | 2 as _"
assert case(S.MatchOr(patterns=[alternatives(), captured()])) == "(1 | 2) | (a as b)"
assert case(open_sequence()) == "a," and case(S.MatchSequence(patterns=[open_sequence(), open_sequence()])) == "(a,), (a,)"
assert case(S.MatchSequence(delimiters="()", patterns=[S.MatchAs(name=n("a"))])) == "(a,)"
assert case(S.MatchSequence(delimiters="[]", patterns=[open_sequence()])) == "[(a,)]"
assert case(S.MatchMapping(keys=[u("-", c("1"))], patterns=[open_sequence()])) == "{-1: (a,)}"
assert case(S.MatchSequence(delimiters="()")) == "()" and case(S.MatchSequence(patterns=[S.MatchSequence()])) == "(),"
print("ok")

## PYPRT-06 · Printing validates the tree and checks it against the version

In [ ]:
with raises(PrintError, match="an If needs a test"):
    P(S.If(body=pas()))
with raises(PrintError, match="'class' is a keyword"):
    P(v("class"))
with raises(PrintError, match="TemplateStr needs Python 3.14, but this is Python 3.12"):
    Python312.print(S.TemplateStr(prefix="t", quote="'"))
with raises(PrintError, match="body[0]: Import.is_lazy needs Python 3.15, but this is Python 3.14"):
    Python314.print(S.Module(body=[S.Import(is_lazy=True, names=[S.Alias(name=S.DottedName(names=[n("a")]))])]))
assert Python312.print(S.Module(body=[expr(walrus())])) == "(x := a)\n"
assert P(S.Return(value=walrus())) == "return (x := a)" and P(S.For(target=v("x"), iter=walrus(), body=pas())) == (
    "for x in (x := a):\n    pass")
print("ok")